# Lesson 2 — Permutation Importance
Curso: *Kaggle Machine Learning Explainability*
Formato: Study + Reference + Hands‑on

---

## 1) Objetivos da Lesson

- Entender o que é **feature importance**.
- Compreender o conceito e o funcionamento de **Permutation Importance**.
- Calcular permutation importance em um modelo treinado.
- Interpretar corretamente os valores de importância (incluindo valores negativos).
- Relacionar importância de features com hipóteses sobre o problema (ex.: táxis em Nova York).

## 2) Glossário Técnico

- **Feature Importance:** medida de quanto uma feature contribui para a performance do modelo.
- **Permutation Importance:** técnica que mede a queda de performance ao embaralhar uma feature.
- **Baseline:** performance original do modelo antes de qualquer embaralhamento.
- **Loss / Métrica:** função usada para avaliar o modelo (ex.: MAE, RMSE, accuracy).
- **Validação:** conjunto de dados usado para avaliar o modelo após o treino.

## 3) Mini‑Referência (API Style)

- `pandas.read_csv(path)`
- `DataFrame.query(expr)`
- `train_test_split(X, y, random_state=...)`
- `RandomForestRegressor(n_estimators=..., random_state=...)`
- `RandomForestRegressor.fit(X, y)`
- `RandomForestRegressor.predict(X)`
- `eli5.sklearn.PermutationImportance(estimator, random_state=...)`
- `PermutationImportance.fit(X, y)`
- `eli5.show_weights(perm, feature_names=...)`

## 4) Introdução Conceitual

Uma das perguntas mais naturais após treinar um modelo é:

> Quais features o modelo considera mais importantes?

Essa pergunta é respondida pelo conceito de **feature importance**.

Existem várias formas de medir importância, mas a Lesson 2 foca em **Permutation Importance**, porque ela é:

- rápida de calcular;
- amplamente utilizada;
- consistente com o que esperamos de uma boa medida de importância.

A ideia central é simples:

> se embaralhar uma coluna piora muito a performance, então o modelo dependia fortemente dessa coluna.

## 5) Implementação passo a passo

Nesta lesson, o Kaggle usa um subconjunto do dataset da competição
**New York City Taxi Fare Prediction**. Vamos seguir a mesma lógica,
apenas ajustando o caminho de dados para um ambiente local.

In [ ]:
# 5.1. Imports principais
import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

print("Bibliotecas carregadas.")

In [ ]:
# 5.1.2. Definindo PROJECT_ROOT e caminho dos dados
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "nyc_taxi_train.csv")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)

In [ ]:
# 5.1.3. Carregando os dados (amostra, como no Kaggle)
# No Kaggle: '../input/new-york-city-taxi-fare-prediction/train.csv', nrows=50000
data = pd.read_csv(DATA_PATH, nrows=50000)

print("Shape bruto:", data.shape)
data.head()

In [ ]:
# 5.2.1. Removendo outliers de coordenadas e tarifas negativas
data = data.query(
    'pickup_latitude > 40.7 and pickup_latitude < 40.8 and '
    'dropoff_latitude > 40.7 and dropoff_latitude < 40.8 and '
    'pickup_longitude > -74 and pickup_longitude < -73.9 and '
    'dropoff_longitude > -74 and dropoff_longitude < -73.9 and '
    'fare_amount > 0'
)

print("Shape após filtros:", data.shape)
data.head()

In [ ]:
# 5.2.2. Definindo target e features base
y = data["fare_amount"]

base_features = [
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "passenger_count",
]

X = data[base_features]

X.head()

In [ ]:
# 5.2.3. Dividindo em treino e validação
train_X, val_X, train_y, val_y = train_test_split(
    X, y, random_state=1
)

train_X.shape, val_X.shape

In [ ]:
# 5.3.1. Modelo base: RandomForestRegressor
first_model = RandomForestRegressor(
    n_estimators=50,
    random_state=1
).fit(train_X, train_y)

print("Modelo treinado.")

In [ ]:
# 5.3.2. Avaliando o modelo com MAE (apenas para referência)
val_preds = first_model.predict(val_X)
baseline_mae = mean_absolute_error(val_y, val_preds)

print(f"MAE de validação (baseline): {baseline_mae:.3f}")

In [ ]:
# 5.4.1. Importando eli5 e PermutationImportance
import eli5
from eli5.sklearn import PermutationImportance

In [ ]:
# 5.4.2. Calculando permutation importance no modelo base
perm = PermutationImportance(
    first_model,
    random_state=1
).fit(val_X, val_y)

print("Permutation Importance calculado.")

In [ ]:
# 5.4.3. Visualizando os pesos de importância
eli5.show_weights(perm, feature_names=val_X.columns.tolist())

Interpretação:

- As features no topo são as que mais pioram a performance quando embaralhadas.
- O valor numérico indica quanto a métrica (score interno) se deteriora.
- O valor após `±` indica a variabilidade entre diferentes embaralhamentos.

In [ ]:
# 5.5.1. Criando features de distância
data["abs_lon_change"] = abs(data["dropoff_longitude"] - data["pickup_longitude"])
data["abs_lat_change"] = abs(data["dropoff_latitude"] - data["pickup_latitude"])

features_2 = [
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "abs_lat_change",
    "abs_lon_change",
]

X2 = data[features_2]

In [ ]:
# 5.5.2. Novo split com as novas features
new_train_X, new_val_X, new_train_y, new_val_y = train_test_split(
    X2, y, random_state=1
)

new_train_X.shape, new_val_X.shape

In [ ]:
# 5.5.3. Novo modelo com features de distância
second_model = RandomForestRegressor(
    n_estimators=30,
    random_state=1
).fit(new_train_X, new_train_y)

print("Segundo modelo treinado.")

In [ ]:
# 5.5.4. Permutation Importance no segundo modelo
perm2 = PermutationImportance(
    second_model,
    random_state=1
).fit(new_val_X, new_val_y)

eli5.show_weights(perm2, feature_names=new_val_X.columns.tolist())

Interpretação típica (como na lesson):

- As features de distância (`abs_lat_change`, `abs_lon_change`) tendem a aparecer como mais importantes.
- Isso sugere que **distância percorrida** é mais relevante que a posição exata de pickup/dropoff.
- Ainda assim, as coordenadas absolutas mantêm alguma importância (efeitos de localização).

## 6) Observações pedagógicas do Copilot

- O Kaggle assume que você já está confortável com:
  - leitura de CSV com `pandas`;
  - divisão treino/validação com `train_test_split`;
  - treino de modelos com `RandomForestRegressor`.

- A grande novidade aqui não é o modelo, mas **como usamos o modelo**:
  - o modelo já está treinado;
  - não alteramos pesos internos;
  - apenas embaralhamos colunas de validação e medimos o impacto.

- Pontos importantes:
  - Permutation Importance depende da **métrica** usada (MAE, RMSE, accuracy, etc.).
  - Valores negativos de importância podem aparecer por acaso, especialmente em datasets pequenos.
  - Features correlacionadas podem “dividir” importância entre si.

- O exemplo dos táxis em Nova York mostra como:
  - usar importância de features para gerar hipóteses;
  - criar novas features (distâncias) para testar essas hipóteses;
  - verificar se o modelo passa a depender mais dessas novas features.

## 7) Conclusão

Nesta lesson, você:

- viu o conceito de **feature importance** na prática;
- aprendeu a calcular **Permutation Importance** com `eli5`;
- interpretou importâncias em um problema real (táxis em Nova York);
- viu como importância de features pode inspirar novas features (distâncias);
- reforçou a ideia de que explicabilidade é uma ferramenta de:
  - debugging,
  - geração de hipóteses,
  - comunicação com stakeholders.

Na próxima etapa do curso de Explainability, você verá técnicas que vão além de
“quão importante é uma feature” e começam a responder:

> **Como** essa feature afeta as previsões?

Isso será feito com **Partial Dependence Plots (PDPs)**.